In [6]:
!pip install -q sentence-transformers faiss-cpu gradio
import faiss, numpy as np
from sentence_transformers import SentenceTransformer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 74.3 MB/s eta 0:00:00


In [ ]:
import os
os.makedirs("data", exist_ok=True)
services = {"income_certificate": """Service: Income Certificate (आय प्रमाण पत्र)
Documents: Aadhaar card, ration card, address proof, self-declaration, passport photo
Process: Apply on MP Online portal or at a Lok Seva Kendra, upload documents, pay fee, track status
Time: About 7 days
Link: https://mponline.gov.in""",

"domicile_certificate": """Service: Domicile Certificate (मूल निवासी प्रमाण पत्र)
Documents: Aadhaar card, address proof, school certificate, passport photo
Process: Apply on MP Online portal or at a Lok Seva Kendra, upload documents, track status
Time: About 7 to 15 days
Link: https://mponline.gov.in""",

"caste_certificate": """Service: Caste Certificate (जाति प्रमाण पत्र)
Documents: Aadhaar card, family caste record or father's certificate, address proof, passport photo
Process: Apply on MP Online portal or at a Lok Seva Kendra, submit documents, verification by officer
Time: About 15 days
Link: https://mponline.gov.in""",
}

for name, text in services.items():
  with open(f"data/{name}.txt", "w", encoding="utf-8") as f:
    f.write(text)
    print(os.listdir("data"))

['income_certificate.txt']
['domicile_certificate.txt', 'income_certificate.txt']
['domicile_certificate.txt', 'income_certificate.txt', 'caste_certificate.txt']


In [13]:
from google.colab import userdata
import os
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [25]:
import glob, os, numpy as np, faiss
from sentence_transformers import SentenceTransformer


os.makedirs("data", exist_ok=True)

services = {
"income_certificate": """Service: Income Certificate (आय प्रमाण पत्र)
Documents: Aadhaar card, ration card, address proof, self-declaration, passport photo
Process: Apply on MP Online portal or at a nearby Lok Seva Kendra, upload documents, pay fee, track status
Official link: https://mponline.gov.in""",

"domicile_certificate": """Service: Domicile Certificate (मूल निवासी प्रमाण पत्र)
Documents: Aadhaar card, address proof, school or birth certificate, photo
Process: Apply on MP Online portal or at a Lok Seva Kendra
Official link: https://mponline.gov.in""",
}

for name, text in services.items():
    with open(f"data/{name}.txt", "w", encoding="utf-8") as f:
        f.write(text)


model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


print(f"Current working directory: {os.getcwd()}")
print(f"Does 'data' directory exist: {os.path.exists('data')}")
if os.path.exists("data"):
    print(f"Contents of 'data' directory: {os.listdir('data')}")

files = sorted(glob.glob("data/*.txt"))
print(f"Files found by glob: {files}")
docs = [open(f, encoding="utf-8").read() for f in files]
print(f"Number of documents loaded: {len(docs)}")


embeddings = model.encode(docs)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))


def search(question, k=1):
    q = model.encode([question])
    _, ids = index.search(np.array(q), k)
    return [docs[i] for i in ids[0]]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Current working directory: /content
Does 'data' directory exist: True
Contents of 'data' directory: ['domicile_certificate.txt', 'income_certificate.txt']
Files found by glob: ['data/domicile_certificate.txt', 'data/income_certificate.txt']
Number of documents loaded: 2


In [14]:
from google import genai
client = genai.Client()

In [15]:
from google import genai

# Use the 'model', 'docs', and 'index' defined in ZTVWiBVPZMHO
# The 'model' (SentenceTransformer instance) is already defined and encoded 'docs' into 'embeddings'
# and created 'index' in cell ZTVWiBVPZMHO.

client = genai.Client()

def answers(question, k=2):
  # Use the 'model' from ZTVWiBVPZMHO for encoding questions
  q = model.encode([question])
  # Use the 'index' and 'docs' from ZTVWiBVPZMHO
  _, idx = index.search(np.array(q), min(k, len(docs)))
  context = "\n\n".join([docs[i] for i in idx[0]])
  prompt = f"""You are a helpful government service assistant.
Answer ONLY using the context below. Reply in the same language as the user.
If the answer is not in the context, say you don't know and suggest visiting the official portal.

context:
{context}

Question: {question}"""
  resp = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
  return resp.text

In [26]:
import gradio as gr

def chat(message, history):
  return answers(message)

  gr.ChatInterface(
      chat,
      tittle="🏛️ नागरिक सेवा सहायक | Citizen Service Assistant",
      textbox=gr.Textbox(placeholder="अपना सवाल पूछें / Ask your question"),
      ).launch(share=True, debug=True)


In [27]:
print(search("mujhey aay parman patr banwana hai"[0]))
print("_ _ _ _ _ _")

['Service: Income Certificate (आय प्रमाण पत्र)\nDocuments: Aadhaar card, ration card, address proof, self-declaration, passport photo\nProcess: Apply on MP Online portal or at a nearby Lok Seva Kendra, upload documents, pay fee, track status\nOfficial link: https://mponline.gov.in']
_ _ _ _ _ _


In [28]:
from google.colab import userdata
import os
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [33]:
import time
from google import genai
client = genai.Client()

MODELS = ["gemini-3.8-flash", "gemini-flash-latest"]   # tried in this order

def ask_gemini(prompt, retries=3):
    for model_name in MODELS:
        for attempt in range(retries):
            try:
                resp = client.models.generate_content(model=model_name, contents=prompt)
                return resp.text
            except Exception as e:
                print(f"{model_name} attempt {attempt+1} failed: {str(e)[:80]}")
                time.sleep(2 * (attempt + 1))
    return "सर्वर अभी व्यस्त है, कृपया कुछ देर बाद पुनः प्रयास करें। /Server is busy, please try again in a moment."

def answer(question, k=2):
    context = "\n\n".join(search(question, k=min(k, len(docs))))
    prompt = f"""You are a helpful government service assistant.
Answer ONLY using the context below. Reply in the same language as the user.
If the answer is not in the context, say you don't know and suggest visiting the official portal.

Context:
{context}

Question: {question}"""
    return ask_gemini(prompt)

print(answer("What documents do I need for an income certificate?"))

gemini-3.8-flash attempt 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
gemini-3.8-flash attempt 2 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
gemini-3.8-flash attempt 3 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
gemini-flash-latest attempt 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
gemini-flash-latest attempt 2 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
gemini-flash-latest attempt 3 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
सर्वर अभी व्यस्त है, कृपया कुछ देर बाद पुनः प्रयास करें। /Server is busy, please try again in a moment.


In [ ]:
!pip install -q gradio
import gradio as gr

def chat(message, history):
  return answers(message)
gr.ChatInterface(chat, title="🏛️ नागरिक सेवा सहायक | Citizen Service Assistant").launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a48b2952ede4c58c95.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 882, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 410, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2330, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1688, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/gradio/utils.py", line 1081, in async_wrapper
    response = await 